# Arlians — Train · Analyze · Probe · Run (Emergence Loop)

A **self-contained validation loop** for the reward design: it trains a founding cohort,
then shows whether the *right* things happened. Runs as-is on a fresh Kaggle GPU notebook
(it clones the repo + installs deps itself) or locally from the repo root.

The reward only encodes **innate drives** (survival + homeostasis + the urge to reproduce).
Farming, building, storing, division of labor, a self-sustaining population — **never
rewarded**. If they rise over training they *emerged* as instrumentally-optimal survival
behavior. That is the whole thesis of the innate-drive backbone.

Stages:
1. **Train** — a Stage-1 founding-cohort run (`train/run.py`, no respawn → a real cohort that
   must survive). Writes `ckpt.pt` + `metrics.jsonl`.
2. **Dashboard** — behavioral time-series. Watch the ★EMERGENT panels climb while never being
   rewarded; watch drives stay healthy and births-vs-lifespan stay balanced.
3. **Probe** — trained-vs-random gates (survival / farming / persistence).
4. **Sim run** — render the *resultant model* acting in the world (a GIF).

> **Scale note.** A short local/MPS run validates the *machinery and early trends* (drives
> stabilizing, lifespan rising, balanced births, reward climbing). **Full emergence** (farming
> %, building actually rising) needs the long GPU run — thousands of updates at 256–512 scale.
> Use the `QUICK` preset locally; raise to the FULL preset on a Kaggle GPU (Accelerator → GPU,
> Internet → On).

## 0. Configuration

In [ ]:
import os

ON_KAGGLE = os.path.isdir("/kaggle/working")

# --- repo source (used only if the repo isn't already importable, e.g. fresh Kaggle notebook) ---
GITHUB_USER   = "adit-rah"
GITHUB_REPO   = "arlians"
GITHUB_BRANCH = "main"
GITHUB_TOKEN  = ""            # only needed for a private repo

# --- where the repo lives / where outputs go ---
if ON_KAGGLE:
    REPO_DIR = "/kaggle/working/arlians"
    OUT_DIR  = "/kaggle/working"
else:
    REPO_DIR = "."            # local: run from the repo root (or point this at it)
    OUT_DIR  = "runs"

CKPT_PATH    = os.path.join(OUT_DIR, "ckpt_emergence.pt")
METRICS_PATH = os.path.join(OUT_DIR, "metrics_emergence.jsonl")

# --- what to do ---
DO_TRAIN = True              # train a fresh run here, then analyze it. False = analyze existing outputs.
FRESH    = True              # start from scratch (REQUIRED for the new reward objective — don't warm-start old weights)
WORLD_SEED = 42

# --- scale preset ---  (QUICK = local sanity; FULL = Kaggle GPU emergence run)
PRESET = "FULL" if ON_KAGGLE else "QUICK"
PRESETS = {
    # width, slot-cap, founding cohort, updates, horizon
    "QUICK": dict(width=64,  agents=256,  init_agents=64,  updates=120,  horizon=48),
    "FULL":  dict(width=256, agents=2048, init_agents=256, updates=4000, horizon=128),
}
P = PRESETS[PRESET]
print("kaggle:", ON_KAGGLE, "| preset:", PRESET, P)
print("repo:", REPO_DIR, "| out:", OUT_DIR)

## 1. Setup

Makes the repo importable. On Kaggle it clones the repo and installs the few non-torch deps;
locally it just adds the repo root to the path. Idempotent — safe to re-run.

In [ ]:
import sys, subprocess

def _has_repo(d):
    return os.path.isdir(os.path.join(d, "world")) and os.path.isdir(os.path.join(d, "sim"))

# Clone the repo if it isn't already present (fresh Kaggle notebook).
if not _has_repo(REPO_DIR):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    else:
        print("cloning", url, "->", REPO_DIR)
        subprocess.check_call(["git", "clone", "-b", GITHUB_BRANCH, url, REPO_DIR])

os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
os.makedirs(OUT_DIR, exist_ok=True)

# Deps: Kaggle ships torch; the sim needs numpy/scipy/opensimplex; matplotlib/imageio for plots+gif.
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy>=1.26", "scipy>=1.12", "opensimplex>=0.4", "imageio", "matplotlib"],
                   check=False)

import importlib
for m in ["world", "sim.simulation", "train.run", "train.ppo"]:
    importlib.import_module(m)

import torch
print("repo importable from", os.getcwd())
print("cuda:", torch.cuda.is_available(),
      "| gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 2. Train (founding cohort → grow a civilization)

Stage-1 run: a modest cohort with **`--no-respawn`** so the population isn't artificially
topped up — lifespan and a self-sustaining population become real signals. Re-run with a
higher `updates` (and `FRESH=False`, which adds `--resume`) to continue across Kaggle commits;
or set `DO_TRAIN=False` to skip straight to analysis of an existing run.

For a long Kaggle job use **Save Version → Save & Run All (Commit)** so `/kaggle/working`
outputs (`ckpt_emergence.pt`, `metrics_emergence.jsonl`) persist.

In [ ]:
if DO_TRAIN:
    cmd = [
        sys.executable, "train/run.py",
        "--width", str(P["width"]), "--seed", str(WORLD_SEED),
        "--agents", str(P["agents"]), "--init-agents", str(P["init_agents"]),
        "--updates", str(P["updates"]), "--horizon", str(P["horizon"]),
        "--no-respawn",                                  # real cohort: no population crutch
        "--reset-floor", str(max(1, P["init_agents"] // 4)),
        "--checkpoint", CKPT_PATH, "--checkpoint-every", "20",
        "--log", METRICS_PATH,
    ]
    if not FRESH:
        cmd.append("--resume")
    print(" ".join(cmd), "\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:                              # stream training output live
        print(line, end="")
    proc.wait()
    print("\n[done] exit", proc.returncode)
else:
    print("DO_TRAIN=False — skipping training; will analyze existing outputs at", METRICS_PATH)

## 3. Load metrics

In [ ]:
import json

rows = []
with open(METRICS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(len(rows), "updates logged")

def series(key):
    """Series over updates; checks top-level row then the nested 'behavior' dict."""
    return [r[key] if key in r else r.get("behavior", {}).get(key) for r in rows]

def action_series(name):
    """Per-action fraction over updates (0 where unused)."""
    return [r.get("behavior", {}).get("action_frac", {}).get(name, 0.0) for r in rows]

updates = series("update")
print("behavior keys:", sorted((rows[-1].get("behavior") or {}).keys()) if rows else [])

## 4. Emergence dashboard

Panels tagged **★ EMERGENT** are behaviors the reward never pays for — if they trend up, the
backbone is producing civilization on its own. **births vs lifespan** is the breed-to-death
watch (births spiking while mean age craters ⇒ lower `w_r_birth`).

In [ ]:
import matplotlib.pyplot as plt

def _clean(xs, ys):
    xy = [(a, b) for a, b in zip(xs, ys) if b is not None]
    return ([a for a, _ in xy], [b for _, b in xy]) if xy else ([], [])

fig, ax = plt.subplots(3, 3, figsize=(16, 12))

def plot1(a, key, title, ylab, emergent=False):
    x, y = _clean(updates, series(key))
    a.plot(x, y, lw=1.6)
    a.set_title(("★ EMERGENT — " if emergent else "") + title, fontsize=10)
    a.set_xlabel("update"); a.set_ylabel(ylab); a.grid(alpha=0.3)

plot1(ax[0,0], "pct_farmed", "calories from crops", "fraction", emergent=True)

x, y = _clean(updates, series("structures_built"))
ax[0,1].plot(x, y, lw=1.6, label="built/rollout")
xs, ys = _clean(updates, series("structures_peak"))
if ys: ax[0,1].plot(xs, ys, lw=1.2, alpha=0.7, label="peak standing")
ax[0,1].set_title("★ EMERGENT — structures", fontsize=10)
ax[0,1].set_xlabel("update"); ax[0,1].set_ylabel("count"); ax[0,1].grid(alpha=0.3); ax[0,1].legend(fontsize=8)

for k, lbl in [("mean_energy","energy"),("mean_hydration","hydration"),("mean_thermal","thermal")]:
    x, y = _clean(updates, series(k))
    ax[0,2].plot(x, y, lw=1.4, label=lbl)
ax[0,2].set_title("drive satisfaction (the instinct)", fontsize=10)
ax[0,2].set_xlabel("update"); ax[0,2].set_ylabel("level"); ax[0,2].set_ylim(0,1); ax[0,2].grid(alpha=0.3); ax[0,2].legend(fontsize=8)

x, yb = _clean(updates, series("births"))
ax[1,0].plot(x, yb, color="tab:green", lw=1.5)
ax[1,0].set_ylabel("births", color="tab:green")
a2 = ax[1,0].twinx()
xa, ya = _clean(updates, series("mean_age"))
a2.plot(xa, ya, color="tab:red", lw=1.5)
a2.set_ylabel("mean_age", color="tab:red")
ax[1,0].set_title("births vs lifespan (breed-to-death watch)", fontsize=10)
ax[1,0].set_xlabel("update"); ax[1,0].grid(alpha=0.3)

for k in ["mean_reward","mean_return"]:
    x, y = _clean(updates, series(k))
    ax[1,1].plot(x, y, lw=1.5, label=k)
ax[1,1].set_title("RL objective", fontsize=10)
ax[1,1].set_xlabel("update"); ax[1,1].set_ylabel("value"); ax[1,1].grid(alpha=0.3); ax[1,1].legend(fontsize=8)

for k in ["pop_mean","n_living"]:
    x, y = _clean(updates, series(k))
    if y: ax[1,2].plot(x, y, lw=1.5, label=k)
ax[1,2].set_title("population", fontsize=10)
ax[1,2].set_xlabel("update"); ax[1,2].set_ylabel("agents"); ax[1,2].grid(alpha=0.3); ax[1,2].legend(fontsize=8)

plot1(ax[2,0], "specialization_index", "division of labor", "mean pairwise JS", emergent=True)
plot1(ax[2,1], "signal_action_mi", "signal↔action MI (proto-language)", "bits", emergent=True)

for name in ["FORAGE","EAT","DRINK","PLANT","HARVEST","BUILD","REPRODUCE","MOVE"]:
    y = action_series(name)
    if any(v > 0.01 for v in y):
        x, yy = _clean(updates, y)
        ax[2,2].plot(x, yy, lw=1.3, label=name)
ax[2,2].set_title("action mix drift", fontsize=10)
ax[2,2].set_xlabel("update"); ax[2,2].set_ylabel("fraction"); ax[2,2].grid(alpha=0.3); ax[2,2].legend(fontsize=7, ncol=2)

plt.tight_layout(); plt.show()

**How to read it**

- **★ pct_farmed / structures / specialization / signal-MI rising** = emergence (none are in the
  reward). They're the *last* thing to appear, after basic homeostasis is solved — near 0 early
  is expected, especially on the QUICK preset.
- **drive satisfaction** is the instinct the reward *does* pay for — it should stay well above 0.
  Energy pinned near 0 = stuck at the food-economy wall (give it the FULL preset + more updates).
- **births vs lifespan**: healthy = both rise together. Births spiking while mean_age falls =
  breed-to-death → lower `w_r_birth`.

## 5. Probe gates (trained vs random)

`scripts/probe.py` rolls the trained policy and a random baseline on the SAME world with **no
respawn**, and checks survival / farming / persistence. All three passing = emergence confirmed.

In [ ]:
if not os.path.exists(CKPT_PATH):
    print("No checkpoint at", CKPT_PATH, "— train first (DO_TRAIN=True).")
else:
    cmd = [sys.executable, "scripts/probe.py", "--checkpoint", CKPT_PATH,
           "--size", str(P["width"]), "--agents", str(P["init_agents"]),
           "--max-agents", str(P["agents"]), "--steps", "720"]
    print(" ".join(cmd), "\n")
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 6. Sim run — watch the *resultant model* act

Render the trained policy moving through the world (`scripts/demo.py`, same seed/size as
training, `--no-respawn` so you watch a real cohort instead of slots re-seeding at random
tiles). The printed summary includes `pct_calories_farmed`, `specialization_index`, and
`signal_action_mi` for this rollout.

In [ ]:
from IPython.display import Image as IPyImage, display

DEMO_OUT = os.path.join(OUT_DIR, "resultant_model.gif")
if not os.path.exists(CKPT_PATH):
    print("No checkpoint at", CKPT_PATH, "— train first (DO_TRAIN=True).")
else:
    cmd = [sys.executable, "scripts/demo.py",
           "--checkpoint", CKPT_PATH,
           "--size", str(P["width"]), "--seed", str(WORLD_SEED),
           "--agents", str(P["init_agents"]), "--max-agents", str(P["agents"]),
           "--steps", "300", "--render-every", "1", "--no-respawn",
           "--out", DEMO_OUT]
    print(" ".join(cmd), "\n")
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
    if os.path.exists(DEMO_OUT):
        display(IPyImage(filename=DEMO_OUT))